# Yield-curve paper reproduction

Independent implementation of [arXiv:2608.07536v1](https://arxiv.org/html/2608.07536v1).

Tree count and model scope are recorded in the manifest below. The supplied replication uses 50 trees versus the paper's 500. Exact author-code replication is limited by missing specifications; see `references/REPRODUCTION_DECISIONS.md`.

In [ ]:
from pathlib import Path
import sys, json
import pandas as pd
import matplotlib.pyplot as plt
ROOT = Path.cwd()
if not (ROOT / 'src').exists():
    raise RuntimeError('Run this notebook from the extracted project folder.')
sys.path.insert(0, str(ROOT / 'src'))
RESULTS = ROOT / 'results/replication_50trees'
manifest = json.loads((RESULTS / 'manifest.json').read_text())
manifest['config']

## Input and benchmark fidelity

The original data reproduce the Table 1 statistics. The recovered origin schedule and AR indexing match the benchmark results at displayed precision.

In [ ]:
summary = pd.read_csv(RESULTS / 'summary_statistics_comparison.csv')
comparison = pd.read_csv(RESULTS / 'paper_comparison.csv', dtype={'target': str})
comparison.groupby(['model', 'metric']).agg(
    rounded_matches=('rounds_to_paper', 'sum'),
    cells=('rounds_to_paper', 'size'),
    mean_absolute_difference=('difference', lambda s: s.abs().mean())
)

## Forecast results

RMSE is in percentage points. Relative RMSE below one beats the random walk. These are recomputed values, separate from the paper's reference tables.

In [ ]:
metrics = pd.read_csv(RESULTS / 'forecast_metrics.csv', dtype={'target': str})
metrics.query('horizon == 1').pivot(index='model', columns='target', values='relative_rmse').round(3)

In [ ]:
from IPython.display import Image, display
display(Image(filename=str(RESULTS / 'figures/relative_rmse.png')))

## Compare macro augmentation

Paired DM tests use aligned dates and HAC uncertainty. A publication lag is implemented; revised macro snapshots are not real-time data vintages.

In [ ]:
paired = pd.read_csv(RESULTS / 'paired_model_comparisons.csv', dtype={'target': str})
paired[(paired['model'] == 'Direct-RF-X') & (paired['benchmark'] == 'Direct-RF')].sort_values(['horizon','target'])

In [ ]:
display(Image(filename=str(RESULTS / 'figures/macro_importance.png')))

## Curve fitting and trading

Trading uses synthetic aged bonds, gross-normalized DV01-neutral slope positions, and overlapping holding periods. It is an independent backtest reconstruction; its returns do not reproduce the paper's undisclosed normalization exactly.

In [ ]:
pd.read_csv(RESULTS / 'curve_fit_comparison.csv')

In [ ]:
pd.read_csv(RESULTS / 'trading_summary.csv').sort_values(['trade_type','average_rank_total'])

## Rerun

Use the terminal in this project folder:

```bash
python -m pip install -r requirements.txt
python run.py --config configs/full_500_trees.json --output results/full_500 --workers 8
python make_report.py --results results/full_500
```

The full run can take several hours. Optional recursive validation is in `tune.py`. For transformations matching official FRED-MD rather than Appendix A, use `configs/official_fred_sensitivity.json`.